# Fink/LSST — Dipole Analysis per Deep Drilling Field


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-20
- last update : 2026-05-21 (add direction plots per DDF; fix date axis on time plots)

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → inline")

In [ ]:
FINK_API = "https://api.lsst.fink-portal.org"
CONE_RADIUS = 3600.0
N_MAX = 5000

DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

NB_TAG = "DIPOLES_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str):
    """Save figure to both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Column definitions

In [ ]:
DIPOLE_COLS = [
    "r:isDipole",
    "r:isNegative",
    "r:dipoleFitAttempted",
    "r:dipoleFluxDiff",
    "r:dipoleFluxDiffErr",
    "r:dipoleMeanFlux",
    "r:dipoleMeanFluxErr",
    "r:dipoleLength",
    "r:dipoleAngle",
    "r:dipoleNdata",
    "r:dipoleChi2",
]
FLUX_COLS = [
    "r:ra",
    "r:dec",
    "r:midpointMjdTai",
    "r:band",
    "r:diaObjectId",
    "r:diaSourceId",
    "r:psfFlux",
    "r:psfFluxErr",
    "r:scienceFlux",
    "r:scienceFluxErr",
    "r:templateFlux",
    "r:templateFluxErr",
    "r:apFlux",
    "r:apFluxErr",
    "r:nDiaSources",
    "r:extendedness",
    "r:reliability",
    "r:visit",
    "r:detector",
    "r:x",
    "r:y",
]
CROSSMATCH_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_gaiadr3_Plx",
    "f:xm_gaiadr3_e_Plx",
    "f:xm_gaiadr3_PhotGMag",
    "f:xm_simbad_otype",
    "f:xm_legacydr8_pstar",
    "f:xm_legacydr8_zphot",
    "f:xm_legacydr8_fqual",
    "f:xm_tns_fullname",
    "f:xm_tns_type",
    "f:xm_vsx_Type",
    "f:xm_gcvs_type",
    "f:xm_mangrove_2MASS_name",
    "f:xm_mangrove_HyperLEDA_name",
    "f:clf_cats_class",
    "f:clf_cats_score",
    "f:clf_snnSnVsOthers_score",
    "f:is_sso",
]
ALL_COLS = FLUX_COLS + DIPOLE_COLS + CROSSMATCH_COLS
COLUMNS_STR = ",".join(ALL_COLS)
print(f"Total columns: {len(ALL_COLS)}")

## 3. API wrappers

In [ ]:
def _post_json(url: str, payload: dict, timeout: int = 1000) -> list | dict:
    """POST a JSON payload and return the parsed response, raising on HTTP errors."""
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_conesearch(
    ra: float,
    dec: float,
    radius: float,
    n: int = N_MAX,
    columns: str | None = COLUMNS_STR,
) -> pd.DataFrame:
    """
    Cone search via /api/v1/conesearch — returns one row per alert (diaSource).
    Column names must use the 'r:' or 'f:' prefix.
    """
    payload = {"ra": ra, "dec": dec, "radius": radius, "n": n, "output-format": "json"}
    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload)
        return pd.DataFrame(raw) if raw else pd.DataFrame()
    except Exception as e:
        print(f"fetch_conesearch ERROR (ra={ra:.3f}, dec={dec:.3f}): {e}")
        return pd.DataFrame()


print("API helpers defined.")

## 4. Fetch alerts per DDF and save to parquet

In [ ]:
FORCE_RELOAD = False
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name, (ra, dec) in DEEP_FIELDS.items():
    parquet_path = os.path.join(DIR_DATA, f"{field_name}_alerts.parquet")

    if os.path.exists(parquet_path) and not FORCE_RELOAD:
        df = pd.read_parquet(parquet_path)
        print(f"[{field_name:12s}] Loaded from cache: {len(df):6d} alerts")
    else:
        print(f"[{field_name:12s}] Fetching from API  (ra={ra:.4f}, dec={dec:.4f}) …", end=" ")
        t0 = time.time()
        df = fetch_conesearch(ra, dec, radius=CONE_RADIUS, n=N_MAX)
        elapsed = time.time() - t0
        if df.empty:
            print(f"→ NO DATA  ({elapsed:.1f}s)")
            ddf_alerts[field_name] = df
            continue
        df["field"] = field_name
        for bool_col in ["r:isDipole", "r:isNegative", "r:dipoleFitAttempted"]:
            if bool_col in df.columns:
                df[bool_col] = (
                    df[bool_col]
                    .map(
                        lambda v: (
                            True
                            if str(v).strip().lower() in ("true", "1", "yes")
                            else (False if str(v).strip().lower() in ("false", "0", "no") else pd.NA)
                        )
                    )
                    .astype("boolean")
                )
        for col in [
            "r:psfFlux",
            "r:psfFluxErr",
            "r:scienceFlux",
            "r:scienceFluxErr",
            "r:templateFlux",
            "r:templateFluxErr",
            "r:apFlux",
            "r:apFluxErr",
            "r:dipoleFluxDiff",
            "r:dipoleMeanFlux",
            "r:dipoleLength",
            "r:dipoleAngle",
            "r:dipoleChi2",
            "r:midpointMjdTai",
            "r:ra",
            "r:dec",
        ]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        df.to_parquet(parquet_path, index=False)
        print(f"→ {len(df):6d} alerts  ({elapsed:.1f}s) → saved")
    ddf_alerts[field_name] = df

print("\nFetch complete.")

## 5. Summary statistics per DDF

In [ ]:
rows = []
for field_name, df in ddf_alerts.items():
    if df.empty:
        rows.append(
            {
                "field": field_name,
                "n_alerts": 0,
                "n_dipoles": 0,
                "dipole_fraction": np.nan,
                "n_fit_attempted": 0,
            }
        )
        continue
    n_total = len(df)
    n_dipoles = int(df["r:isDipole"].fillna(False).sum()) if "r:isDipole" in df.columns else 0
    n_fit = int(df["r:dipoleFitAttempted"].fillna(False).sum()) if "r:dipoleFitAttempted" in df.columns else 0
    rows.append(
        {
            "field": field_name,
            "n_alerts": n_total,
            "n_fit_attempted": n_fit,
            "n_dipoles": n_dipoles,
            "dipole_fraction": n_dipoles / n_total if n_total > 0 else np.nan,
            "fit_fraction": n_fit / n_total if n_total > 0 else np.nan,
        }
    )
df_summary = pd.DataFrame(rows).set_index("field")
print("\nDipole summary per DDF:")
print(df_summary.to_string(float_format="{:.4f}".format))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
fields = df_summary.index.tolist()
fracs = df_summary["dipole_fraction"].fillna(0).values * 100.0
colors = plt.cm.tab10(np.linspace(0, 1, len(fields)))
bars = ax.bar(fields, fracs, color=colors, edgecolor="k", linewidth=0.5)
ax.set_ylabel("Dipole fraction (%)")
ax.set_title("Fraction of DIA alerts flagged as dipoles per DDF")
ax.tick_params(axis="x", rotation=30)
for bar, frac in zip(bars, fracs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f"{frac:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8,
    )
plt.tight_layout()
savefig("dipole_fraction_per_ddf")
plt.show()

## 6. Sky map of dipoles per DDF

For each DDF we plot:
- **grey dots**: all alerts (non-dipole)
- **coloured arrows**: dipole alerts — arrow direction = `dipoleAngle`, length ∝ `dipoleLength`, colour by band


In [ ]:
def plot_dipole_skymap(df: pd.DataFrame, field_name: str, max_arrows: int = 3000):
    """
    Sky map with quiver arrows showing dipole position angles.
    Arrow direction = dipoleAngle PA (N=up, E=left in RA-inverted axes).
    Arrow length is normalised to be visually clear regardless of dipoleLength.
    """
    if df.empty:
        print(f"[{field_name}] No data — skipping.")
        return
    required = {"r:ra", "r:dec", "r:isDipole"}
    if required - set(df.columns):
        print(f"[{field_name}] Missing columns — skipping.")
        return

    df = df.copy()
    df["r:isDipole"] = df["r:isDipole"].fillna(False).astype(bool)
    df_nd = df[~df["r:isDipole"]]
    df_dp = df[df["r:isDipole"]]

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        df_nd["r:ra"].values,
        df_nd["r:dec"].values,
        s=1,
        c="grey",
        alpha=0.3,
        label=f"non-dipole ({len(df_nd):,})",
    )

    if len(df_dp) > 0 and "r:dipoleAngle" in df_dp.columns and "r:dipoleLength" in df_dp.columns:
        df_plot = df_dp.dropna(subset=["r:ra", "r:dec", "r:dipoleAngle", "r:dipoleLength"])
        if len(df_plot) > max_arrows:
            df_plot = df_plot.sample(max_arrows, random_state=42)
            print(f"[{field_name}] Downsampled to {max_arrows} arrows.")

        angle_rad = np.radians(df_plot["r:dipoleAngle"].values)
        length_deg = df_plot["r:dipoleLength"].values / 3600.0
        scale = 0.05 / np.nanmedian(length_deg) if np.nanmedian(length_deg) > 0 else 1.0
        dx = np.sin(angle_rad) * length_deg * scale
        dy = np.cos(angle_rad) * length_deg * scale

        if "r:band" in df_plot.columns:
            for band, grp in df_plot.groupby("r:band"):
                idx = grp.index
                ax.quiver(
                    grp["r:ra"].values,
                    grp["r:dec"].values,
                    dx[df_plot.index.get_indexer(idx)],
                    dy[df_plot.index.get_indexer(idx)],
                    color=BAND_COLORS.get(band, "red"),
                    angles="xy",
                    scale_units="xy",
                    scale=0.2,
                    width=0.002,
                    headwidth=4,
                    headlength=4,
                    label=f"dipole band={band} ({len(grp):,})",
                    alpha=0.7,
                )
        else:
            ax.quiver(
                df_plot["r:ra"].values,
                df_plot["r:dec"].values,
                dx,
                dy,
                color="red",
                angles="xy",
                scale_units="xy",
                scale=0.2,
                width=0.002,
                headwidth=4,
                headlength=4,
                label=f"dipole ({len(df_plot):,})",
                alpha=0.7,
            )
    else:
        ax.scatter(
            df_dp["r:ra"].values,
            df_dp["r:dec"].values,
            s=8,
            c="red",
            alpha=0.5,
            label=f"dipole ({len(df_dp):,})",
        )

    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_aspect("auto", "box")
    ax.set_title(
        f"Dipole sky map — {field_name}\n"
        f"total={len(df):,}  dipoles={len(df_dp):,}  "
        f"({100 * len(df_dp) / max(1, len(df)):.1f}%)"
    )
    ax.invert_xaxis()
    ax.legend(loc="best", fontsize=7, markerscale=3)
    plt.tight_layout()
    savefig(f"skymap_dipoles_{field_name.replace('-', '_')}")
    plt.show()


print("plot_dipole_skymap() defined.")

In [ ]:
for field_name, df in ddf_alerts.items():
    plot_dipole_skymap(df, field_name)

## 6b. Dipole direction (angle) distribution per DDF

For each DDF two panels are shown side by side:

- **Left — polar rose diagram**: distribution of `r:dipoleAngle` (position angle of the
  dipole axis, 0° = North, 90° = East, clockwise convention).  
  A **uniform distribution** (dashed red circle) indicates random noise orientation.  
  Any **preferred direction** would betray a systematic shift at the telescope level
  (tracking error, read-out direction, wind shake, …).
- **Right — linear histogram by band**: the same angles split by filter, to check
  whether some bands have a specific preferred orientation.

The `dipoleAngle` column follows the standard astronomical position angle convention.


In [ ]:
def plot_dipole_direction_per_ddf(
    ddf_alerts: dict,
    angle_col: str = "r:dipoleAngle",
    band_col: str = "r:band",
    n_bins_rose: int = 36,
) -> None:
    """
    For each DDF plot two panels side by side:
    Left  — polar rose diagram of dipoleAngle (all dipole alerts combined).
    Right — linear histogram of dipoleAngle split by filter band.

    Parameters
    ----------
    ddf_alerts   : dict  field_name → DataFrame
    angle_col    : column with dipole PA in degrees (0–360)
    band_col     : column with filter band label
    n_bins_rose  : angular bins in the rose diagram (default 36 = 10°/bin)
    """
    fields_with_data = [
        (name, df) for name, df in ddf_alerts.items() if not df.empty and angle_col in df.columns
    ]
    if not fields_with_data:
        print("No dipoleAngle data available.")
        return

    n_fields = len(fields_with_data)
    fig = plt.figure(figsize=(12, 3.8 * n_fields))
    bin_edges_rad = np.linspace(0, 2 * np.pi, n_bins_rose + 1)
    bin_centers = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    bin_width = 2 * np.pi / n_bins_rose

    for row, (field_name, df) in enumerate(fields_with_data):
        # Restrict to dipole-flagged alerts when the flag column is available
        if "r:isDipole" in df.columns:
            df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
        else:
            df_dip = df.copy()

        angles_deg = pd.to_numeric(df_dip[angle_col], errors="coerce").dropna().values % 360.0
        angles_rad = np.radians(angles_deg)

        # ── Left: polar rose diagram ──────────────────────────────────────────
        ax_pol = fig.add_subplot(n_fields, 2, 2 * row + 1, projection="polar")
        if len(angles_rad) > 0:
            counts_rose, _ = np.histogram(angles_rad, bins=bin_edges_rad)
            freq = counts_rose / counts_rose.sum() if counts_rose.sum() > 0 else counts_rose
            ax_pol.bar(
                bin_centers,
                freq,
                width=bin_width * 0.92,
                bottom=0,
                color="steelblue",
                edgecolor="white",
                linewidth=0.3,
                alpha=0.85,
            )
            # Dashed circle at the uniform level
            uniform_level = 1.0 / n_bins_rose
            ax_pol.plot(
                np.linspace(0, 2 * np.pi, 300),
                np.full(300, uniform_level),
                "--",
                color="crimson",
                lw=1.0,
                alpha=0.8,
                label="uniform",
            )
            ax_pol.legend(loc="lower right", fontsize=6, bbox_to_anchor=(1.25, -0.05))

        ax_pol.set_theta_zero_location("N")  # North at top
        ax_pol.set_theta_direction(-1)  # clockwise (PA convention)
        ax_pol.tick_params(labelsize=7)
        ax_pol.set_title(
            f"{field_name} — rose diagram (n={len(angles_deg):,})",
            va="bottom",
            pad=18,
            fontsize=9,
        )
        # Cardinal labels
        for pa_deg, label in [(0, "N"), (90, "E"), (180, "S"), (270, "W")]:
            ax_pol.text(
                np.radians(pa_deg),
                ax_pol.get_rmax() * 1.2,
                label,
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
            )

        # ── Right: linear histogram by band ──────────────────────────────────
        ax_lin = fig.add_subplot(n_fields, 2, 2 * row + 2)
        lin_edges = np.linspace(0, 360, n_bins_rose + 1)
        if len(angles_deg) == 0:
            ax_lin.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax_lin.transAxes)
        elif band_col in df_dip.columns:
            band_order = [b for b in list("ugrizy") if b in df_dip[band_col].dropna().unique()]
            bottom = np.zeros(n_bins_rose)
            for band in band_order:
                sub = (
                    pd.to_numeric(df_dip.loc[df_dip[band_col] == band, angle_col], errors="coerce")
                    .dropna()
                    .values
                    % 360.0
                )
                if len(sub) == 0:
                    continue
                cnts, _ = np.histogram(sub, bins=lin_edges)
                ax_lin.bar(
                    lin_edges[:-1],
                    cnts,
                    width=360.0 / n_bins_rose * 0.9,
                    bottom=bottom,
                    color=BAND_COLORS.get(band, "grey"),
                    edgecolor="white",
                    linewidth=0.3,
                    label=f"{band} (n={len(sub):,})",
                    alpha=0.85,
                )
                bottom += cnts
        else:
            cnts, _ = np.histogram(angles_deg, bins=lin_edges)
            ax_lin.bar(
                lin_edges[:-1],
                cnts,
                width=360.0 / n_bins_rose * 0.9,
                color="steelblue",
                edgecolor="white",
                linewidth=0.3,
            )

        ax_lin.set_xlabel("Dipole PA (deg, N through E)")
        ax_lin.set_ylabel("N dipole alerts")
        ax_lin.set_xlim(0, 360)
        ax_lin.set_xticks(np.arange(0, 361, 45))
        ax_lin.set_xticklabels(
            ["N\n0°", "45°", "E\n90°", "135°", "S\n180°", "225°", "W\n270°", "315°", "N\n360°"],
            fontsize=7,
        )
        ax_lin.set_title(f"{field_name} — dipoleAngle by band", fontsize=9)
        ax_lin.legend(loc="upper right", fontsize=7, ncol=2)

    fig.suptitle("Dipole orientation per DDF", fontsize=11, y=1.005)
    plt.tight_layout()
    savefig("dipole_direction_per_ddf")
    plt.show()


plot_dipole_direction_per_ddf(ddf_alerts)

## 7. Dipole fraction vs time (per DDF)

We bin alerts by MJD interval (configurable `MJD_BIN_DAYS`) and compute for each bin
`n_dipoles / n_total`.  
A **top x-axis** shows human-readable calendar dates (YYYY-MM-DD, inclined 40°),
computed from the MJD values via `astropy.time.Time`.  
A **right y-axis** (grey bars) shows the total number of alerts per bin.


### Utility functions

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40° to the left for readability.

    The tick positions share the same data coordinate as the primary MJD axis, so
    zooming/panning the primary axis will keep the dates aligned.

    Parameters
    ----------
    ax         : primary matplotlib Axes (MJD on x)
    mjd_values : array of MJD values present in the current plot
                 (used to determine the range and spacing)
    n_ticks    : target number of date tick marks (capped at the available range)
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    # Evenly-spaced tick positions in MJD coordinates
    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)  # list of 'YYYY-MM-DD'

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())  # must match the primary axis limits exactly
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

In [ ]:
MJD_BIN_DAYS = 1  # bin width in days


def compute_dipole_fraction_vs_time(
    df: pd.DataFrame,
    bin_days: float = MJD_BIN_DAYS,
    mjd_col: str = "r:midpointMjdTai",
    flag_col: str = "r:isDipole",
) -> pd.DataFrame:
    """
    Bin alerts by MJD and compute the dipole fraction in each bin.

    Returns columns: mjd_bin_center, n_total, n_dipoles,
                     dipole_fraction, dipole_fraction_err
    """
    if df.empty or mjd_col not in df.columns or flag_col not in df.columns:
        return pd.DataFrame()
    df = df.copy()
    df[flag_col] = df[flag_col].fillna(False).astype(bool)
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")
    df = df.dropna(subset=[mjd_col])
    if df.empty:
        return pd.DataFrame()
    mjd_min, mjd_max = df[mjd_col].min(), df[mjd_col].max()
    bins = np.arange(mjd_min, mjd_max + bin_days, bin_days)
    df["mjd_bin"] = pd.cut(df[mjd_col], bins=bins, labels=False)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    rows = []
    for i, center in enumerate(bin_centers):
        sub = df[df["mjd_bin"] == i]
        n_tot = len(sub)
        n_dip = int(sub[flag_col].sum())
        frac = n_dip / n_tot if n_tot > 0 else np.nan
        frac_e = np.sqrt(n_dip) / n_tot if (n_tot > 0 and n_dip > 0) else np.nan
        rows.append(
            {
                "mjd_bin_center": center,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": frac,
                "dipole_fraction_err": frac_e,
            }
        )
    return pd.DataFrame(rows)


print("compute_dipole_fraction_vs_time() defined.")

In [ ]:
# ── Plot dipole fraction vs time for each DDF ─────────────────────────────────
fig, axes = plt.subplots(
    nrows=len(DEEP_FIELDS),
    ncols=1,
    figsize=(11, 3.2 * len(DEEP_FIELDS)),
    sharex=False,
)
if len(DEEP_FIELDS) == 1:
    axes = [axes]

for ax, (field_name, df) in zip(axes, ddf_alerts.items()):
    df_time = compute_dipole_fraction_vs_time(df, bin_days=MJD_BIN_DAYS)

    if df_time.empty:
        ax.set_title(f"{field_name} — no data")
        continue

    mjd = df_time["mjd_bin_center"].values
    frac = df_time["dipole_fraction"].values * 100.0
    err = df_time["dipole_fraction_err"].fillna(0).values * 100.0

    # ── Main series: dipole fraction ─────────────────────────────────────────
    ax.errorbar(
        mjd,
        frac,
        yerr=err,
        fmt="o-",
        ms=4,
        lw=1.2,
        capsize=3,
        color="steelblue",
        label=f"bin={MJD_BIN_DAYS}d",
    )

    # ── Right y-axis: N alerts per bin (grey bars) ────────────────────────────
    ax_r = ax.twinx()
    ax_r.bar(
        mjd,
        df_time["n_total"].values,
        width=MJD_BIN_DAYS * 0.8,
        color="lightgrey",
        alpha=0.4,
        label="N alerts",
    )
    ax_r.set_ylabel("N alerts", fontsize=7, color="grey")
    ax_r.tick_params(axis="y", labelcolor="grey", labelsize=7)

    ax.set_ylabel("Dipole fraction (%)")
    ax.set_title(f"{field_name} — dipole fraction vs time (bin={MJD_BIN_DAYS} days)")
    ax.set_xlabel("MJD (TAI)")
    ax.legend(loc="upper left", fontsize=7)

    # ── Top x-axis: human-readable calendar dates (YYYY-MM-DD, inclined 40°) ─
    # Uses add_date_axis_on_top() which calls astropy Time for MJD→date.
    add_date_axis_on_top(ax, mjd, n_ticks=7)

plt.tight_layout()
savefig(f"dipole_fraction_vs_time_bin{MJD_BIN_DAYS}d")
plt.show()

## 8. Dipole fraction per visit (per DDF)

Per-visit statistics: fraction of alerts with `isDipole=True` in each visit,
ordered by visit ID (which encodes the date `YYYYMMDDNNNNN`).


In [ ]:
def compute_dipole_fraction_per_visit(
    df: pd.DataFrame,
    visit_col: str = "r:visit",
    flag_col: str = "r:isDipole",
    mjd_col: str = "r:midpointMjdTai",
    band_col: str = "r:band",
) -> pd.DataFrame:
    """Return one row per (visit, band) pair with the dipole fraction."""
    if df.empty or visit_col not in df.columns or flag_col not in df.columns:
        return pd.DataFrame()
    df = df.copy()
    df[flag_col] = df[flag_col].fillna(False).astype(bool)
    group_cols = [visit_col] + ([band_col] if band_col in df.columns else [])
    agg = (
        df.groupby(group_cols)
        .agg(
            n_total=(flag_col, "count"),
            n_dipoles=(flag_col, "sum"),
            mjd_mean=(mjd_col, "mean") if mjd_col in df.columns else (flag_col, "count"),
        )
        .reset_index()
    )
    agg["dipole_fraction"] = agg["n_dipoles"] / agg["n_total"]
    return agg.sort_values(visit_col).reset_index(drop=True)


print("compute_dipole_fraction_per_visit() defined.")

In [ ]:
for field_name, df in ddf_alerts.items():
    df_visit = compute_dipole_fraction_per_visit(df)
    if df_visit.empty:
        print(f"[{field_name}] No per-visit data — skipping.")
        continue

    fig, ax = plt.subplots(figsize=(12, 4))
    if "r:band" in df_visit.columns:
        for band, grp in df_visit.groupby("r:band"):
            ax.plot(
                grp["r:visit"].astype(str),
                grp["dipole_fraction"] * 100,
                "o-",
                ms=3,
                lw=1,
                color=BAND_COLORS.get(band, "grey"),
                label=f"band={band}",
            )
    else:
        ax.plot(
            df_visit["r:visit"].astype(str),
            df_visit["dipole_fraction"] * 100,
            "o-",
            ms=3,
            lw=1,
            color="steelblue",
        )
    ax.xaxis.set_major_locator(plt.MaxNLocator(20))
    ax.tick_params(axis="x", rotation=45, labelsize=6)
    ax.set_xlabel("Visit ID")
    ax.set_ylabel("Dipole fraction (%)")
    ax.set_title(f"{field_name} — dipole fraction per visit")
    ax.legend(loc="best", fontsize=7)
    plt.tight_layout()
    savefig(f"dipole_fraction_per_visit_{field_name.replace('-', '_')}")
    plt.show()

## 9. Dipole morphology distributions

Distributions of `dipoleLength`, `dipoleAngle`, `dipoleChi2`, and `dipoleFluxDiff`
for all DDFs combined.


In [ ]:
frames = []
for field_name, df in ddf_alerts.items():
    if df.empty or "r:isDipole" not in df.columns:
        continue
    df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
    df_dip["field"] = field_name
    frames.append(df_dip)

df_all_dipoles = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Total dipole alerts across all DDFs: {len(df_all_dipoles):,}")
if not df_all_dipoles.empty:
    print(df_all_dipoles[["field"]].value_counts().to_string())

In [ ]:
if df_all_dipoles.empty:
    print("No dipole data — skipping morphology plots.")
else:
    morph_cols = {
        "r:dipoleLength": ("Dipole length (arcsec)", 0, 20, 50),
        "r:dipoleAngle": ("Dipole angle (deg)", 0, 360, 36),
        "r:dipoleChi2": ("Dipole chi2", 0, 50, 50),
        "r:dipoleFluxDiff": ("Dipole flux diff (nJy)", None, None, 60),
    }
    fig, axes = plt.subplots(1, len(morph_cols), figsize=(14, 4))
    for ax, (col, (xlabel, xmin, xmax, nbins)) in zip(axes, morph_cols.items()):
        if col not in df_all_dipoles.columns:
            ax.set_title(f"{col}\n(not found)")
            continue
        vals = pd.to_numeric(df_all_dipoles[col], errors="coerce").dropna().values
        if xmin is not None and xmax is not None:
            vals = vals[(vals >= xmin) & (vals <= xmax)]
        ax.hist(vals, bins=nbins, color="steelblue", edgecolor="white", linewidth=0.3)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("N alerts")
        ax.set_title(f"{col.split(':')[1]}\n(n={len(vals):,})")
    plt.suptitle("Dipole morphology — all DDFs combined", y=1.02, fontsize=11)
    plt.tight_layout()
    savefig("dipole_morphology_all_ddfs")
    plt.show()

In [ ]:
# ── dipoleAngle polar histogram (rose diagram) — all DDFs combined ────────────
if df_all_dipoles.empty or "r:dipoleAngle" not in df_all_dipoles.columns:
    print("Skipping polar histogram — no dipoleAngle data.")
else:
    angles_deg = pd.to_numeric(df_all_dipoles["r:dipoleAngle"], errors="coerce").dropna().values
    angles_rad = np.radians(angles_deg % 360)
    n_bins = 36
    bin_edges = np.linspace(0, 2 * np.pi, n_bins + 1)
    counts, _ = np.histogram(angles_rad, bins=bin_edges)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="polar")
    ax.bar(
        bin_centers,
        counts,
        width=2 * np.pi / n_bins * 0.9,
        color="steelblue",
        edgecolor="white",
        linewidth=0.4,
        alpha=0.8,
    )
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(f"Dipole angle distribution\n(all DDFs, n={len(angles_deg):,})", va="bottom", pad=20)
    plt.tight_layout()
    savefig("dipole_angle_polar_histogram")
    plt.show()

## 10. Dipole fraction per band and per DDF

In [ ]:
band_rows = []
for field_name, df in ddf_alerts.items():
    if df.empty or "r:isDipole" not in df.columns or "r:band" not in df.columns:
        continue
    df = df.copy()
    df["r:isDipole"] = df["r:isDipole"].fillna(False).astype(bool)
    for band, grp in df.groupby("r:band"):
        n_tot = len(grp)
        n_dip = int(grp["r:isDipole"].sum())
        band_rows.append(
            {
                "field": field_name,
                "band": band,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": n_dip / n_tot if n_tot > 0 else np.nan,
            }
        )

df_band = pd.DataFrame(band_rows)
if not df_band.empty:
    pivot = df_band.pivot_table(index="field", columns="band", values="dipole_fraction").reindex(
        columns=list("ugrizy")
    )
    print("Dipole fraction per field × band:")
    print(pivot.to_string(float_format="{:.4f}".format))

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.values * 100, aspect="auto", cmap="YlOrRd", vmin=0)
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns.tolist())
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index.tolist())
    ax.set_xlabel("Band")
    ax.set_ylabel("DDF")
    ax.set_title("Dipole fraction (%) per DDF × band")
    plt.colorbar(im, ax=ax, label="Dipole fraction (%)")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val * 100:.1f}", ha="center", va="center", fontsize=8)
    plt.tight_layout()
    savefig("dipole_fraction_field_vs_band_heatmap")
    plt.show()

## 11. Save concatenated dipole catalogue

In [ ]:
if not df_all_dipoles.empty:
    out_path = os.path.join(DIR_DATA, "all_ddfs_dipoles_only.parquet")
    df_all_dipoles.to_parquet(out_path, index=False)
    print(f"Saved {len(df_all_dipoles):,} dipole alerts → {out_path}")
    display(df_all_dipoles.head(5))
else:
    print("No dipole data — nothing saved.")

## 12. Next steps

1. `02_dipole_cutouts.ipynb` — cutouts via `/api/v1/cutouts`
2. `03_dipole_sources.ipynb` — full diaSource LC via `/api/v1/sources`
3. `04_dipole_butler_cross.ipynb` — Butler visit cross-match at USDF
4. `05_dipole_psf_analysis.ipynb` — dipoleLength vs PSF FWHM per visit
